# 手写 LIML 估计程序

> 对应代码：`MathStatsCode/code_in_notes/Appendix.B/MLE_liml.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：用 ml model 实现有限信息极大似然（LIML）：联立结构方程与第一阶段方程，并让两个方程的协方差参与似然。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Appendix.B`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Appendix.B")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

`clear`、`set more off` 准备环境。本 notebook 用 ml 实现**有限信息极大似然（LIML）**估计（附录中的进阶内容）。


In [ ]:
%%stata
clear
set more off


**LIML by hand**

这一段定义两个程序（`limlobj` 似然与 `liml` 外层命令），核心思想是把结构方程与第一阶段方程放进**同一个似然**中：
- 模型声明 `ml model lf1 limlobj (Structual: y = x) (First_stage: endo = z) /rho`：第一个方程是结构方程（被解释变量 y 对内生变量与控制变量），第二个方程是第一阶段（内生变量对工具变量与控制变量），最后 ` /rho` 是额外标量参数；
- `mleval theta1=b, eq(1)`、`theta2=b, eq(2)` 分别取出两个方程的**线性指数**（用各自系数与变量计算出的拟合值）；
- `resid2 = ML_y2 - theta2`：内生变量减去第一阶段拟合值 = **第一阶段残差**（内生变量中未被工具变量解释的部分）；
- `resid1 = ML_y1 - theta1 - theta3*resid2`：结构误差扣除与第一阶段残差相关的部分——
- 把两个残差的标准差 `sigma1`、`sigma2` 用各自的样本标准差直接代入（相当于把尺度参数剖析掉），对数似然为两个独立正态密度的和：`-lnσ1-lnσ2-resid1²/(2σ1²)-resid2²/(2σ2²)`；
- `lf1` 评估器要求提供解析梯度：`g1=resid1/σ1²`（对结构方程线性指数求导）、`g2=-θ3·resid1/σ1²+resid2/σ2²`（对第一阶段线性指数求导）、`g3=resid1·resid2/σ1²`（对 ρ 求导）；
- 外层程序 `liml` 用最小二乘估计值作为初始值（结构方程与第一阶段的 OLS 系数、ρ 的初值取0），再 `ml maximize` 执行优化。这种“控制函数”形式的参数化，让 ρ 捕捉内生性：ρ≠0 说明结构误差与第一阶段残差相关，即内生变量确实内生。


In [ ]:
%%stata
cap program drop limlobj
program limlobj
	args todo b lnfj g1 g2 g3
	tempvar theta1 theta2 theta3 resid2 resid1
	tempname sigma1 sigma2
	mleval `theta1'=`b', eq(1)
	mleval `theta2'=`b', eq(2)
	mleval `theta3'=`b', eq(3)
	quietly{
		gen `resid1'=.
		gen `resid2'=.
		replace `resid2'=$ML_y2 - `theta2'
		replace `resid1'=$ML_y1 - `theta1' - `theta3'*`resid2'
		su `resid1'
		scalar `sigma1'=r(sd)
		su `resid2'
		scalar `sigma2'=r(sd)
		replace `lnfj'=-log(`sigma1')-log(`sigma2') /*
			*/-`resid1'^2/(`sigma1'^2*2) /*
			*/-`resid2'^2/(`sigma2'^2*2)
		if (`todo'==0) exit
		replace `g1'=1/(`sigma1'^2)*`resid1'
		replace `g2'=-1/(`sigma1'^2)*`resid1'*`theta3'/*
			*/+1/(`sigma2'^2)*`resid2'
		replace `g3'=1/(`sigma1'^2)*`resid1'*`resid2'	
	}
end

cap program drop liml
program liml, eclass
	version 15.0
	// endo: 内生变量，至多一个
	// instru: 工具变量，control：控制变量
	// endofunc: 内生变量的函数
	syntax varlist(max=1), endo(varlist max=1) /*
		*/ instru(varlist) control(varlist) [endofc(varlist)]
	tempname dep x z b0 b1 initb
	local `dep' "`varlist'"
	local `x' "`endo' `endofc' `control'"
	local `z' "`instru' `control'"
	ml model lf1 limlobj (Structual:``dep'' = ``x'') /*
		*/ (First_stage:`endo' = ``z'') /rho
	//使用最小二乘估计作为初始值
	quietly{
		reg ``dep'' ``x''
		mat `b0'=e(b)
		reg `endo' ``z''
		mat `b1'=e(b)
		mat `initb'=(`b0', `b1', 0)
		ml init `initb', copy
	}
	ml maximize
end

set obs 1000


**generate data**

生成测试数据：1000个观测，`z` 为工具变量，`u` 为共同冲击，`x1=z+u` 内生，`x2=z+0.5ε` 控制变量，`y=2+x1+x2+u`（结构系数真值都是1）。


In [ ]:
%%stata
gen z=rnormal()
gen u=rnormal()
gen x1=z+rnormal()+u
gen x2=z+0.5*rnormal()
gen y=x1+x2+u


**官方LIML**

先用**官方命令** `ivregress liml y (x1=z) x2` 估计，作为对照基准（同时可与 2SLS 的结果对比——过度识别时两者一般不同，本例恰好识别时数值相同）。


In [ ]:
%%stata
// 最简单情况，与2SLS等价
ivregress liml  y (x1=z) x2


**手写LIML**

再用自编的 `liml y, endo(x1) instru(z) control(x2)` 估计并对比：结构方程中 x1、x2 的系数应接近官方结果（约1.0），ρ 的估计显著为正（对应数据生成中 x1 与 u 的相关性），第一阶段中 z 的系数也应接近1。这个例子把 LIML 的似然构造、参数化与 ml 编程完整地串了起来，是全书最综合的编程练习之一。


In [ ]:
%%stata
liml y, endo(x1) instru(z) control(x2)
